# 04 - Pairwise feature experiments

This notebook evaluates the production feature engine on blocked training pairs. TF-IDF is fitted only on training records and reused for validation. Ground truth is used for analysis/evaluation, never as a feature.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if not SRC.exists(): SRC = PROJECT_ROOT / 'code' / 'business_entity_resolution' / 'src'
sys.path.insert(0, str(SRC.resolve()))
from config import default_config
from data_loader import load_training_data
from preprocessing import preprocess_sources
from blocking import generate_candidates
from features import fit_feature_transformers, transform_features
from validation import build_training_pairs, create_validation_split, evaluate_predictions
from model import train_model, predict_proba

CONFIG = default_config().resolved(PROJECT_ROOT)
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports' / 'features'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MAX_PAIRS = 250_000


In [ ]:
training = load_training_data(CONFIG)
sources = preprocess_sources(training.sources, CONFIG.schema, CONFIG.normalization)
source_names = list(sources)
reference_source = source_names[0]
reference = sources[reference_source]
candidate_sources = {source: sources[source] for source in source_names[1:]}
block_result = generate_candidates(reference, candidate_sources, CONFIG, reference_source=reference_source)
pairs = block_result.pairs.head(MAX_PAIRS).copy()
labeled = build_training_pairs(pairs, training.ground_truth)
print('Blocked pairs:', len(block_result.pairs), 'pairs analyzed:', len(pairs))
print('Positive pairs:', int(labeled.target.sum()), 'negative pairs:', int((labeled.target == 0).sum()))

In [ ]:
# Fit once on records available to the training experiment.
transformers = fit_feature_transformers(reference, candidate_sources, CONFIG)
feature_result = transform_features(pairs, reference, candidate_sources, transformers, CONFIG)
X = feature_result.matrix.toarray()
feature_names = list(feature_result.feature_names)
feature_frame = pd.DataFrame(X, columns=feature_names)
feature_frame['target'] = labeled['target'].to_numpy()
print('Feature matrix:', X.shape)
display(feature_frame.head())

In [ ]:
# Range, NaN/inf, and positive-vs-negative distribution checks.
summary_rows = []
for name in feature_names:
    values = feature_frame[name]
    positive = values[feature_frame.target == 1]
    negative = values[feature_frame.target == 0]
    summary_rows.append({'feature': name, 'dtype': str(values.dtype), 'min': float(values.min()) if len(values) else 0.0, 'max': float(values.max()) if len(values) else 0.0, 'mean': float(values.mean()) if len(values) else 0.0, 'missing_count': int(values.isna().sum()), 'infinite_count': int(np.isinf(values).sum()), 'positive_mean': float(positive.mean()) if len(positive) else np.nan, 'negative_mean': float(negative.mean()) if len(negative) else np.nan, 'mean_difference': float(positive.mean() - negative.mean()) if len(positive) and len(negative) else np.nan})
feature_summary = pd.DataFrame(summary_rows)
display(feature_summary)

In [ ]:
# Correlation and simple redundancy diagnostics.
feature_correlations = feature_frame[feature_names].corr(method='spearman')
high_corr = []
for i, left in enumerate(feature_names):
    for right in feature_names[i + 1:]:
        value = feature_correlations.loc[left, right]
        if pd.notna(value) and abs(value) >= 0.95:
            high_corr.append({'feature_left': left, 'feature_right': right, 'spearman_correlation': float(value)})
display(pd.DataFrame(high_corr))
print('Ground-truth columns excluded from features:', set(feature_frame.columns) & set(training.ground_truth.columns))

In [ ]:
# A measured validation experiment for feature families.
split = create_validation_split(reference, training.ground_truth, validation_fraction=CONFIG.validation.validation_fraction, random_state=CONFIG.validation.random_state)
train_ids, valid_ids = set(split.train_ids), set(split.validation_ids)
train_pairs = block_result.pairs[block_result.pairs.reference_entity_id.astype(str).isin(train_ids)].head(MAX_PAIRS).copy()
valid_pairs = block_result.pairs[block_result.pairs.reference_entity_id.astype(str).isin(valid_ids)].head(MAX_PAIRS).copy()
train_labels = build_training_pairs(train_pairs, training.ground_truth)
valid_labels = build_training_pairs(valid_pairs, training.ground_truth)
train_result = transform_features(train_pairs, split.train_reference, candidate_sources, transformers, CONFIG)
valid_result = transform_features(valid_pairs, split.validation_reference, candidate_sources, transformers, CONFIG)
experiment_rows = []
families = {'all': list(train_result.feature_names), 'name': [n for n in train_result.feature_names if n.startswith('name_')], 'address': [n for n in train_result.feature_names if n.startswith('address_')], 'country_combined': [n for n in train_result.feature_names if n.startswith('country_') or n.startswith('name_address')], 'missingness': [n for n in train_result.feature_names if n.startswith('missing_')]}
for family, names in families.items():
    if not names: continue
    indices = [train_result.feature_names.index(name) for name in names]
    model = train_model(train_result.matrix[:, indices], train_labels.target.to_numpy(), CONFIG.model, feature_names=names, random_state=CONFIG.runtime.random_seed)
    scores = predict_proba(model, valid_result.matrix[:, indices])
    scored = valid_pairs.copy(); scored['score'] = scores; scored['predicted_match'] = scores >= CONFIG.inference.probability_threshold
    metrics = evaluate_predictions(scored, training.ground_truth, reference_ids=split.validation_ids)
    experiment_rows.append({'feature_family': family, 'feature_count': len(names), **metrics})
experiment_results = pd.DataFrame(experiment_rows)
display(experiment_results)

In [ ]:
feature_summary.to_csv(REPORT_DIR / 'feature_summary.csv', index=False)
feature_correlations.to_csv(REPORT_DIR / 'feature_correlations.csv')
experiment_results.to_csv(REPORT_DIR / 'feature_experiment_results.csv', index=False)
summary_text = '# Feature experiment summary\n\n'
summary_text += f'- Analyzed {len(feature_frame)} blocked pairs and {len(feature_names)} stable features.\n'
summary_text += f'- Features with NaN values: {int((feature_summary.missing_count > 0).sum())}; features with infinities: {int((feature_summary.infinite_count > 0).sum())}.\n'
summary_text += '- Positive/negative distribution differences are recorded in feature_summary.csv.\n'
summary_text += '- High absolute Spearman correlations are recorded in feature_correlations.csv and should be reviewed for redundancy.\n'
summary_text += '- Feature-family validation results are recorded in feature_experiment_results.csv; no performance improvement is claimed unless measured there.\n'
(REPORT_DIR / 'feature_summary.md').write_text(summary_text, encoding='utf-8')
print('Saved feature reports to', REPORT_DIR)

## Final decision protocol

Enable feature families only when their validation results and collision/missingness behavior justify them. Keep TF-IDF fitting restricted to training records, preserve deterministic feature order from `features.py`, exclude ground-truth columns, and treat the measured experiment table as the evidence for the final configuration.